# 🔴 Solution: AdaLN-Zero

In [ ]:
import torch
import torch.nn as nn

In [ ]:
# ✅ SOLUTION

class AdaLNZero(nn.Module):
    def __init__(self, hidden_dim: int, cond_dim: int):
        super().__init__()
        self.norm = nn.LayerNorm(hidden_dim, elementwise_affine=False)
        
        # MLP for gamma, beta, and gate
        self.cond_mlp = nn.Sequential(
            nn.SiLU(),
            nn.Linear(cond_dim, 3 * hidden_dim)  # gamma, beta, gate
        )
        
        # Zero-initialize the final projection
        nn.init.zeros_(self.cond_mlp[1].weight)
        nn.init.zeros_(self.cond_mlp[1].bias)
        
        self.gate_proj = nn.Linear(cond_dim, hidden_dim)
        nn.init.zeros_(self.gate_proj.weight)
        nn.init.zeros_(self.gate_proj.bias)
    
    def forward(self, x: torch.Tensor, cond: torch.Tensor) -> torch.Tensor:
        x_norm = self.norm(x)
        gamma_beta_gate = self.cond_mlp(cond)
        gamma, beta, gate = gamma_beta_gate.chunk(3, dim=-1)
        
        gamma = gamma.unsqueeze(1)
        beta = beta.unsqueeze(1)
        gate = gate.unsqueeze(1)
        
        return gate * (gamma * x_norm + beta)

In [ ]:
# Verify
adaln = AdaLNZero(hidden_dim=64, cond_dim=128)
x = torch.randn(2, 10, 64)
cond = torch.randn(2, 128)
out = adaln(x, cond)
print(f"Output max (should be ~0): {out.abs().max().item():.6f}")

In [ ]:
from torch_judge import check
check("adaln_zero")